In [1]:
import pandas as pd
import numpy as np

### Funciones

In [2]:
def calcular_error_medicion(escala, vpp):
    """
    Calcula el error de medición en función de la escala de medición (Volts/División).

    Parámetros:
    escala (pd.Series): Serie con las escalas de medición.
    vpp (pd.Series): Serie con los valores de Vpp.

    Retorna:
    pd.Series: Serie con los errores de medición calculados.
    """

    # 1. Definimos las condiciones (en Voltios)
    condiciones = [
        (escala >= 0.002) & (escala <= 0.005),  # 2mV a 5mV -> 4%
        (escala >= 0.010) & (escala <= 10.0),  # 10mV a 10V -> 3%
    ]
    # 2. Definimos los porcentajes correspondientes
    porcentajes = [0.04, 0.03]
    # 3. Asignamos el porcentaje según la escala
    pct_aplicable = np.select(condiciones, porcentajes, default=np.nan)
    # 4. Calculamos el error de medición
    error_vpp = vpp * pct_aplicable
    return error_vpp


In [3]:
def redondear_cifras_significativas(valor, n):
    """
    Redondea un valor a un número específico de cifras significativas.
    """
    if pd.isna(valor) or valor == 0:
        return valor
    # Calcula los decimales necesarios según la magnitud del número
    decimales = n - 1 - int(np.floor(np.log10(abs(valor))))
    return round(valor, decimales)

In [4]:
def redondear_medicion(row, valor_verdadero, incerteza):
    """
    Redondea una medición según su incertidumbre.
    """
    medicion = row[valor_verdadero]
    incertidumbre = row[incerteza]

    if pd.isna(medicion) or pd.isna(incertidumbre) or incertidumbre == 0:
        return medicion

    # Determina a qué posición decimal debemos redondear
    # Ejemplo: para 0.3 -> -log10(0.3) = 0.52 -> floor = 0 -> decimales = 1
    # Ejemplo: para 0.04 -> -log10(0.04) = 1.39 -> floor = 1 -> decimales = 2
    decimales = int(-np.floor(np.log10(abs(incertidumbre))))

    if decimales >= 0:
        return round(medicion, decimales)
    else:
        # Si la incertidumbre está en las decenas/centenas (ej: ±30)
        return round(medicion, decimales)

### Importamos los datos

In [5]:
path = r".\clase2\raw\mediciones_parte2_raw.csv"

In [6]:
df = pd.read_csv(path, sep=";")
df

,Vpp,Vr,Escala generador,Escala receptor
0,1,0.704,2,1
1,2,1.340,2,1
2,3,2.020,2,1
3,4,2.600,2,1
4,5,3.160,2,1
5,6,3.560,2,1
6,7,4.080,2,1
7,8,4.640,2,1
8,9,5.120,2,1
9,10,5.660,2,1


In [7]:
df.shape

(22, 4)

In [9]:
#df = df.drop(columns=["Unnamed: 8"])

In [8]:
df = df.dropna()
df.shape

(22, 4)

In [10]:
df.columns

Index(['Vpp', 'Vr', 'Escala generador', 'Escala receptor'], dtype='str')

### Limpiando datos

In [11]:
df_clean = pd.DataFrame()
df_clean["vpp_emisor"] = df["Vpp"]
df_clean["vpp_receptor"] = df["Vr"]

In [13]:
df_clean.head(5)

,vpp_emisor,vpp_receptor
0,1,0.704
1,2,1.340
2,3,2.020
3,4,2.600
4,5,3.160


In [16]:
1.34*0.03

0.0402

In [14]:
df_clean["vpp_emisor_error"] = calcular_error_medicion(df["Escala generador"], df["Vpp"])
df_clean["vpp_receptor_error"] = calcular_error_medicion(df["Escala receptor"], df["Vr"])

df_clean["vpp_emisor_error"] = df_clean["vpp_emisor_error"].apply(lambda x: redondear_cifras_significativas(x, 1))
df_clean["vpp_receptor_error"] = df_clean["vpp_receptor_error"].apply(lambda x: redondear_cifras_significativas(x, 1))

df_clean["vpp_emisor"] = df_clean.apply(lambda row: redondear_medicion(row, "vpp_emisor", "vpp_emisor_error"), axis=1)
df_clean["vpp_receptor"] = df_clean.apply(lambda row: redondear_medicion(row, "vpp_receptor", "vpp_receptor_error"), axis=1)
df_clean

,vpp_emisor,vpp_receptor,vpp_emisor_error,vpp_receptor_error
0,1.0,0.70,0.03,0.02
1,2.0,1.34,0.06,0.04
2,3.0,2.02,0.09,0.06
3,4.0,2.60,0.10,0.08
4,5.0,3.16,0.10,0.09
5,6.0,3.60,0.20,0.10
6,7.0,4.10,0.20,0.10
7,8.0,4.60,0.20,0.10
8,9.0,5.10,0.30,0.20
9,10.0,5.70,0.30,0.20


In [45]:
#vt_error = np.array([0.01]*10)
#vt_error

In [38]:
name_columns = ["10 Hz", "100 Hz", "1 KHz", "10 KHz", "100 KHz", "1 MHz"]

In [45]:
for i, name in enumerate(name_columns, start=1):
    df_clean[name] = df.iloc[:, i:i+1]

In [47]:
df_clean["vt_error"] = vt_error

In [17]:
df_clean

,vpp_emisor,vpp_receptor,vpp_emisor_error,vpp_receptor_error
0,1.0,0.70,0.03,0.02
1,2.0,1.34,0.06,0.04
2,3.0,2.02,0.09,0.06
3,4.0,2.60,0.10,0.08
4,5.0,3.16,0.10,0.09
5,6.0,3.60,0.20,0.10
6,7.0,4.10,0.20,0.10
7,8.0,4.60,0.20,0.10
8,9.0,5.10,0.30,0.20
9,10.0,5.70,0.30,0.20


### Guardando los datos

In [18]:
path_clean = r".\clase2\clean\mediciones_parte2_clean.csv"

In [19]:
df_clean.to_csv(path_clean, index=False)